In [3]:

import re
import unicodedata
import pandas as pd
from google.colab import files

# =========================================================
# 1) Upload file CSV
# =========================================================
print("Hãy upload file CSV")
uploaded = files.upload()

csv_files = [f for f in uploaded.keys() if f.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("Không tìm thấy file CSV.")

CSV_PATH = csv_files[0]
print("Đang dùng file:", CSV_PATH)

# =========================================================
# 2) Đọc dữ liệu
# =========================================================
df = pd.read_csv(CSV_PATH)

required_cols = ["Clause", "code"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"File CSV thiếu cột bắt buộc: {missing_cols}")

# =========================================================
# 3) Chuẩn hóa text
# =========================================================
def normalize_text(text):
    if pd.isna(text):
        text = ""
    text = str(text)

    text = unicodedata.normalize("NFKC", text)
    text = text.lower()

    text = (
        text.replace("’", "'")
            .replace("`", "'")
            .replace("“", '"')
            .replace("”", '"')
            .replace("–", "-")
            .replace("—", "-")
    )

    text = re.sub(r"\s+", " ", text).strip()
    return text

# =========================================================
# 4) Cụm từ RC / RP
# =========================================================
RC_PHRASES = [
    "recommend",
    "must visit",
    "recommended",
    "highly recommend",
    "totally recommend",
    "definitely recommend",
    "would recommend",
    "really recommend",
    "strongly recommend",
    "highly recommended",
    "definitely recommended",
    "would absolutely recommend",
    "would definitely recommend",
    "would totally recommend",
    "can recommend",
    "can highly recommend",
    "recommend this place",
    "recommend this hotel",
    "recommend this tour",
    "recommend this resort",
    "recommend this beach",
    "recommend this restaurant",
    "recommend this experience",
    "recommend to anyone",
    "recommend to others",
    "recommend to friends",
    "recommend to family",
    "recommend for couples",
    "recommend for families",
    "recommend for solo travelers",
    "recommend for a short stay",
    "recommend for a relaxing stay",
    "worth a visit",
    "worth staying",
    "worth booking",
    "worth the price",
    "worth every penny",
    "well worth a visit",
    "worth visiting",
    "worth staying here",
    "10/10 would recommend",
    "a place i'd recommend",
    "a place i’d recommend",
    "do not recommend",
    "not recommended",
    "wouldn't recommend",
    "wouldn’t recommend",
    "would not recommend",
    "can't recommend",
    "can’t recommend",
    "cannot recommend",
    "hard to recommend",
    "difficult to recommend",
    "wouldn't recommend this place",
    "wouldn’t recommend this place",
    "would not recommend this hotel",
    "would not recommend this tour",
    "do not recommend staying here",
    "do not recommend booking here",
    "cannot recommend this place",
    "wouldn't suggest it",
    "wouldn’t suggest it",
    "wouldn't advise it",
    "wouldn’t advise it",
    "worth it",
    "not worth your time",
    "not worth it",
]

RP_PHRASES = [
    "revisit",
    "visit again",
    "stay again",
    "book again",
    "return again",
    "would come back",
    "would definitely come back",
    "would happily come back",
    "would love to come back",
    "would visit again",
    "would definitely visit again",
    "would stay again",
    "would definitely stay again",
    "will stay again",
    "would book again",
    "would definitely book again",
    "will book again",
    "would return",
    "would definitely return",
    "will return",
    "would go back",
    "will be back",
    "we'll be back",
    "we’ll be back",
    "i'll be back",
    "i’ll be back",
    "come back again",
    "stay here again",
    "book with them again",
    "use this company again",
    "choose this hotel again",
    "choose this tour again",
    "would stay here again",
    "would definitely stay here again",
    "would come back again",
    "would come back for sure",
    "we would come back",
    "we would stay again",
    "we would definitely return",
    "i'd stay here again",
    "i’d stay here again",
    "i'd book again",
    "i’d book again",
    "i'd return in a heartbeat",
    "i’d return in a heartbeat",
    "happy to return",
    "happy to stay again",
    "look forward to coming back",
    "look forward to returning",
    "can't wait to come back",
    "can’t wait to come back",
    "would love to return someday",
    "would revisit",
    "would use their service again",
    "would book another stay here",
    "won't be back",
    "won’t be back",
    "would not return",
    "wouldn't return",
    "wouldn’t return",
    "would not come back",
    "wouldn't come back",
    "wouldn’t come back",
    "would not stay again",
    "wouldn't stay again",
    "wouldn’t stay again",
    "would not book again",
    "wouldn't book again",
    "wouldn’t book again",
    "not coming back",
    "never again",
    "one-time visit",
    "one time visit",
    "not again",
    "would not stay here again",
    "wouldn't stay here again",
    "wouldn’t stay here again",
    "would not visit again",
    "wouldn't visit again",
    "wouldn’t visit again",
    "would not book with them again",
    "wouldn't use them again",
    "wouldn’t use them again",
    "won't be returning",
    "won’t be returning",
    "not a place i'd return to",
    "not a place i’d return to",
    "once was enough",
    "one visit was enough",
    "not worth a second visit",
    "not worth returning to",
    "wouldn't choose this place again",
    "wouldn’t choose this place again",
    "come again",
]
# =========================================================
# 5) Bỏ bớt cụm quá mơ hồ để tránh gán nhầm
#    (ví dụ "great place" chỉ là khen, chưa chắc là recommend)
# =========================================================
RC_REMOVE = {
    "must-see",
    "must-try",
    "must stay",
    "a good choice",
    "great choice",
    "top choice",
    "good option",
    "great place",
    "lovely place",
    "amazing place",
    "great spot",
    "hidden gem",
    "a gem",
    "good value",
    "great value",
    "excellent value",
    "one of the best",
    "one of the nicest places",
    "one of the highlights",
    "exceeded expectations",
    "not worth the money",
    "not worth the price",
    "not worth a visit",
    "not worth staying",
    "not worth booking",
    "skip it",
    "avoid",
    "avoid this place",
    "avoid this hotel",
    "avoid this tour",
    "tourist trap",
    "not value for money",
    "poor value",
    "disappointing overall",
    "not as expected",
    "below expectations",
    "far from worth it",
    "not worth the hype",
    "not worth the hassle",
}

RP_REMOVE = {
    "return",
    "come back",
    "go again",
    "back again",
    "be back",
    "use again",
}

def dedupe_keep_order(lst):
    seen = set()
    out = []
    for x in lst:
        x = normalize_text(x)
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

RC_PHRASES = [x for x in dedupe_keep_order(RC_PHRASES) if x not in RC_REMOVE]
RP_PHRASES = [x for x in dedupe_keep_order(RP_PHRASES) if x not in RP_REMOVE]

# =========================================================
# 5) Compile regex
# =========================================================
def phrase_to_pattern(phrase):
    tokens = phrase.split()
    escaped = [re.escape(tok) for tok in tokens]
    body = r"\s+".join(escaped)
    return re.compile(rf"(?<!\w){body}(?!\w)", flags=re.IGNORECASE)

def compile_patterns(phrases):
    phrases = sorted(phrases, key=lambda x: (-len(x), x))
    return [(p, phrase_to_pattern(p)) for p in phrases]

rc_patterns = compile_patterns(RC_PHRASES)
rp_patterns = compile_patterns(RP_PHRASES)

extra_rc_patterns = [
    ("regex_highly_recommend",
     re.compile(r"(?<!\w)(?:highly|strongly|definitely|totally|really)\s+recommend(?:ed)?(?!\w)", re.I)),
    ("regex_would_recommend",
     re.compile(r"(?<!\w)(?:i|we|you|they)\s+(?:would|'d)\s+recommend(?!\w)", re.I)),
    ("regex_10_10_recommend",
     re.compile(r"(?<!\w)10\s*/\s*10\s+(?:would\s+)?recommend(?!\w)", re.I)),
]

extra_rp_patterns = [
    ("regex_return_positive",
     re.compile(
         r"(?<!\w)"
         r"(?:i|we|they)\s+(?:would|will|'d|'ll)\s+"
         r"(?:definitely\s+|happily\s+|gladly\s+)?"
         r"(?:come\s+back|return|revisit|visit\s+again|stay\s+again|book\s+again|go\s+back|be\s+back)"
         r"(?!\w)", re.I)),
    ("regex_return_negative",
     re.compile(
         r"(?<!\w)"
         r"(?:would(?:\s+not|n't)|will\s+not|won't|never|not)\s+"
         r"(?:come\s+back|return|revisit|visit\s+again|stay\s+again|book\s+again|go\s+back|be\s+back)"
         r"(?!\w)", re.I)),
    ("regex_look_forward_return",
     re.compile(
         r"(?<!\w)"
         r"(?:can(?:not|'t)\s+wait\s+to|look(?:ing)?\s+forward\s+to|happy\s+to|hope\s+to)"
         r"\s+"
         r"(?:come\s+back|return|revisit|visit\s+again|stay\s+again|book\s+again|go\s+back|be\s+back)"
         r"(?!\w)", re.I)),
]

# =========================================================
# 6) Detect code mới
# =========================================================
def find_best_match(text, exact_patterns, extra_patterns):
    best = None

    for rule_name, pattern in exact_patterns + extra_patterns:
        m = pattern.search(text)
        if not m:
            continue

        candidate = {
            "rule_name": rule_name,
            "start": m.start(),
            "length": m.end() - m.start()
        }

        if best is None:
            best = candidate
        else:
            if candidate["start"] < best["start"]:
                best = candidate
            elif candidate["start"] == best["start"] and candidate["length"] > best["length"]:
                best = candidate

    return best

def detect_new_code(clause):
    text = normalize_text(clause)

    rc_match = find_best_match(text, rc_patterns, extra_rc_patterns)
    rp_match = find_best_match(text, rp_patterns, extra_rp_patterns)

    if rc_match is None and rp_match is None:
        return None

    if rc_match is not None and rp_match is not None:
        rc_key = (rc_match["start"], -rc_match["length"])
        rp_key = (rp_match["start"], -rp_match["length"])
        return "RC" if rc_key <= rp_key else "RP"

    if rc_match is not None:
        return "RC"

    return "RP"

# =========================================================
# 7) Thay trực tiếp vào cột code
# =========================================================
new_codes = df["Clause"].apply(detect_new_code)

df["code"] = [
    new_code if new_code is not None else old_code
    for new_code, old_code in zip(new_codes, df["code"])
]

# =========================================================
# 8) Thống kê
# =========================================================
changed_mask = new_codes.notna()

print("=" * 80)
print("TỔNG SỐ DÒNG        :", len(df))
print("SỐ DÒNG BỊ THAY CODE:", int(changed_mask.sum()))
print("ĐỔI THÀNH RC        :", int((new_codes == "RC").sum()))
print("ĐỔI THÀNH RP        :", int((new_codes == "RP").sum()))
print("=" * 80)

print("\nVí dụ một vài dòng đã thay:")
display(
    df.loc[changed_mask, ["Clause", "code"]].head(20)
)

# =========================================================
# 9) Lưu file cuối
# =========================================================
output_path = "full_clauses_labeled_final.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"\nĐã lưu file: {output_path}")
files.download(output_path)

Hãy upload file CSV


Saving full_clauses_labeled.csv to full_clauses_labeled.csv
Đang dùng file: full_clauses_labeled.csv
TỔNG SỐ DÒNG        : 288126
SỐ DÒNG BỊ THAY CODE: 9555
ĐỔI THÀNH RC        : 8835
ĐỔI THÀNH RP        : 720

Ví dụ một vài dòng đã thay:


,Clause,code
14,If you’re searching for a snus shop in Hoi An ...,RC
37,"We had a nice, relaxing day and would recommen...",RC
66,It is very recommended for those who want to r...,RC
86,Highly recommended for a chill day,RC
93,The beach peddlers are fun and great to talk t...,RP
100,This beach is definitely worth visiting if you...,RC
118,We used an bang beach as our base and loved it...,RC
131,Highly recommend renting chairs at Soul Kitche...,RC
155,Recommend come about March till September,RC
204,Highly recommend,RC



Đã lưu file: full_clauses_labeled_final.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>